In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '7'

from dotenv import load_dotenv
from collections.abc import Sequence
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output

import flax.jax_utils as flax_utils
import flax.linen as nn
import grain.python as grain
import jax
import json
import numpy as np
from absl import logging
from connectomics.jax import checkpoint, training
from etils import epath
from orbax import checkpoint as ocp
from connectomics.common import ts_utils
import zapbench.models.util as model_util
from zapbench.ts_forecasting import heads, input_pipeline, train
from zapbench.ts_forecasting.configs import infer, mean, linear, timemix, tsmixer, tide
from zapbench.ts_forecasting.infer_idx import _filter_infer_indices

import logging  # Correct import for Python's logging module

import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science'])

load_dotenv()
PATH = os.getenv("ROOT_PATH")
LOG_PATH = os.getenv("LOG_PATH")

logger = logging.getLogger(__name__)  # Create a logger for this module
logger.setLevel(logging.INFO)

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  # Set all spines invisible
  for spine in ax.spines.values():
    spine.set_visible(False)
  # Hide all ticks and tick labels
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
def _get_checkpoint_step(
    checkpoint_manager: ocp.CheckpointManager,
    selection_strategy: str,
) -> int | None:
  """Returns the checkpoint step to use given a selection strategy.

  Args:
    checkpoint_manager: Checkpoint manager.
    selection_strategy: Checkpoint selection strategy, can be 'early_stopping',
      'best_val_loss', or 'latest'.

  Returns:
    Checkpoint step.
  """
  if selection_strategy == 'early_stopping':
    checkpointed_state = dict(
        early_stop=None,
    )
    checkpointed_state = checkpoint.restore_checkpoint(
        checkpoint_manager,
        state=checkpointed_state,
        step=checkpoint_manager.latest_step(),
    )
    return checkpointed_state['early_stop']['best_step']
  elif selection_strategy == 'best_val_loss':
    checkpointed_state = dict(
        track_best_val_loss_step=None,
    )
    checkpointed_state = checkpoint.restore_checkpoint(
        checkpoint_manager,
        state=checkpointed_state,
        step=checkpoint_manager.latest_step(),
    )
    return checkpointed_state['track_best_val_loss_step']['best_step']
  elif selection_strategy == 'latest':
    return checkpoint_manager.latest_step()
  else:
    raise ValueError(f'Unknown checkpoint selection: {selection_strategy}')


def infer_single_step(
    model: nn.Module,
    head: heads.Head,
    train_state: train.TrainState,
    data_source: grain.RandomAccessDataSource,
    idx: int,
    infer_key: jax.Array,  # pylint: disable=unused-argument
    covariates: Sequence[str] = (),
    covariates_static: jax.Array | None = None,
    with_carry: bool = False,
) -> tuple[jax.Array, jax.Array]:
  """Runs independent inference on each index in the test set.

  Returns:
    prediction: prediction array
    target: target array
  """
  carry = None

  batch = data_source[idx]
  if 'covariates_static' in covariates:
    batch['covariates_static'] = covariates_static

  out = train.pred_step(
      model,
      train_state,
      batch,
      covariates,
      initial_carry=carry,
      return_carry=with_carry,
  )

  if not with_carry:
    dist = head.get_distribution(out)
  else:
    carry, dist = out[0], head.get_distribution(out[1])

  prediction = dist.mode()
  target = batch['timeseries_output']

  return prediction, target


def convert_to_serializable(obj):
    if isinstance(obj, (np.ndarray, jax.Array)):
        return obj.tolist()
    elif isinstance(obj, (np.floating, np.integer)):
        return float(obj)
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    return obj

In [ ]:
train_subject_name = "janelia_pretrain"
exp_workdir = f'{LOG_PATH}/n_steps_32/mean/{train_subject_name}/'
exp_config = model_util.load_config(os.path.join(exp_workdir, 'config.json'))

model = model_util.model_from_config(exp_config)

covariates_static = input_pipeline.get_static_covariates(exp_config)

checkpoint_manager = checkpoint.get_checkpoint_manager(
    exp_workdir,
    item_names=(
        'early_stop',
        'train_state',
        'track_best_val_loss_step',
    ),
)

step = _get_checkpoint_step(checkpoint_manager, 'best_val_loss')

checkpointed_state = dict(
    train_state=None,
)
checkpointed_state = checkpoint.restore_checkpoint(
    checkpoint_manager,
    state=checkpointed_state,
    step=step,
)
train_state = checkpointed_state['train_state']
train_state = train.TrainState(
    **train_state
)
train_state = flax_utils.replicate(train_state)

In [ ]:
infer_subect_names = ["subject_01",
                      "subject_02",
                      "subject_03",
                      "subject_04",
                      "subject_05",
                      "subject_06",
                      "subject_07",
                      "subject_12",
                      "subject_14",
                      "subject_15",
                      "subject_16",
                      "subject_17",
                      "240930_traces",]

In [ ]:
all_infer_metrics = dict()
for infer_subect_name in infer_subect_names:
  try:
    infer_config = infer.get_config()
    config = timemix.get_config(f'dataset_name={infer_subect_name},timesteps_input={exp_config.timesteps_input}')
    config.update(infer_config)

    head = heads.create_head(config)
    rng = training.get_rng(config.seed)
    rng, infer_rng = jax.random.split(rng)
    infer_source = input_pipeline.create_inference_source_with_transforms(config)
    infer_key = jax.random.fold_in(key=infer_rng, data=step)

    infer_metrics_cpu_per_subect_dict = dict()

    context = config.timesteps_input_infer + config.timesteps_output_infer
    prediction_list, target_list = [], []
    full_prediction, full_target, transition_idx = [], [], []
    counter = 0
    for infer_idx_set in config.infer_idx_sets:
      name, idx_list = (infer_idx_set[k] for k in ('name', 'idx_list'))
      idx_list = _filter_infer_indices(idx_list, context)
      infer_metrics = None
      train_state = train.merge_batch_stats(train_state)
      for i, idx in enumerate(idx_list):
        prediction, target = infer_single_step(
            model,
            head,
            flax_utils.unreplicate(train_state),
            infer_source,
            idx,
            infer_key=infer_key,
            covariates=tuple(config.covariates),
            covariates_static=covariates_static,
            with_carry=config.infer_with_carry,
        )
        if f'infer_{name}' in head.metrics:
          metrics_update = head.metrics[
              f'infer_{name}'
          ].single_from_model_output(predictions=prediction, targets=target)
          infer_metrics = (
              metrics_update
              if infer_metrics is None
              else infer_metrics.merge(metrics_update)
          )

      if infer_metrics is not None:
        infer_metrics_cpu = jax.tree.map(np.array, infer_metrics.compute())
      infer_metrics_cpu_per_subect_dict[name] = infer_metrics_cpu
  except Exception as e:
    logger.error("Error in %s: %s", infer_subect_name, e)
    continue
  all_infer_metrics[infer_subect_name] = infer_metrics_cpu_per_subect_dict

In [ ]:
path = f"/mnt/storage/misc/zapbench/inference/n_steps_{exp_config.timesteps_input}/mean/cross_subject_infer_metrics/all_infer_metrics_{train_subject_name}.json"
with open(path, "w") as f:
    json.dump(convert_to_serializable(all_infer_metrics), f)

In [ ]:
infer_subject_names = ["subject_01",
                      "subject_02",
                      "subject_03",
                      "subject_04",
                      "subject_05",
                      "subject_06",
                      "subject_07",
                      "subject_12",
                      "subject_15",
                      "subject_16",
                      "subject_17",
                      "zapbench",]

In [ ]:
mae_matrix = []
train_subject_name = "janelia_pretrain"
for n_steps in [4, 32]:
  for model in ["mean", "linear", "timemix"]:
    path = f"/mnt/storage/misc/zapbench/inference/n_steps_{n_steps}/{model}/cross_subject_infer_metrics/all_infer_metrics_{train_subject_name}.json"
    with open(path.format(train_subject_name=train_subject_name), "r") as f:
      all_infer_metrics_loaded = json.load(f)
      all_maes = []
      for infer_subject_name in all_infer_metrics_loaded.keys():
        if infer_subject_name not in ["subject_13", "subject_14"]:
          subject_level_mae = 0
          for infer_condition in all_infer_metrics_loaded[infer_subject_name].keys():
            subject_level_mae += all_infer_metrics_loaded[infer_subject_name][infer_condition][f"infer_{infer_condition}_mae"]
          all_maes.append(subject_level_mae/len(all_infer_metrics_loaded[infer_subject_name]))
        else:
          continue
    mae_matrix.append(all_maes)
mae_matrix = np.array(mae_matrix)
mae_matrix.shape

In [ ]:
import scienceplots
plt.style.use(['science'])


n_subjects = len(infer_subject_names)

fig, ax = plt.subplots(figsize=(4, 4), dpi=1000)
im = ax.imshow(mae_matrix, cmap='coolwarm')

for i in range(6):
    for j in range(len(infer_subject_names)):
        text = ax.text(j, i, f'{mae_matrix[i, j]:.3f}',
                      ha="center", va="center", color="black", fontsize=4)

ax.set_xticks([])
ax.set_yticks([])

label_names = []
for name in infer_subject_names:
    if name == "zapbench":
        label_names.append("zap")
    else:
        # Extract number if present, else keep as is
        import re
        match = re.search(r'(\d+)', name)
        if match:
            label_names.append(match.group(1))
        else:
            label_names.append(name)
ax.set_xlabel('Test subject', fontsize=6)
ax.set_title('MAE, pre-training across subjects', fontsize=6)
ax.set_xticks(np.arange(len(infer_subject_names)))
ax.set_yticks(np.arange(6))
ax.set_xticklabels(label_names, rotation=0, fontsize=6)
ax.set_yticklabels(["Mean, $h = 4$", "Linear, $h = 4$", "Time-Mix, $h = 4$", "Mean, $h = 32$", "Linear, $h = 32$", "Time-Mix, $h = 32$"], fontsize=6)
ax.tick_params(axis='both', which='both', length=0)


vmin = 0.01
vmax = 0.04
im.set_clim(vmin, vmax)

# Make colorbar the same height as the matrix
from mpl_toolkits.axes_grid1 import make_axes_locatable
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
cbar = plt.colorbar(im, cax=cax)
cbar.ax.tick_params(labelleft=False, left=False, right=False)
cbar.ax.tick_params(labelsize=6)
plt.tight_layout()

In [ ]:
mae_matrix_d = mae_matrix - mae_matrix_32

In [ ]:
n_subjects = len(infer_subject_names)

fig, ax = plt.subplots(figsize=(4, 4), dpi=1000)
im = ax.imshow(mae_matrix_d, cmap='coolwarm')

for i in range(len(infer_subject_names)):
    for j in range(len(infer_subject_names)):
        text = ax.text(j, i, f'{mae_matrix_d[i, j]:.3f}',
                      ha="center", va="center", color="black", fontsize=4)

ax.set_xticks([])
ax.set_yticks([])

label_names = []
for name in infer_subject_names:
    if name == "zapbench":
        label_names.append("zap")
    else:
        # Extract number if present, else keep as is
        import re
        match = re.search(r'(\d+)', name)
        if match:
            label_names.append(match.group(1))
        else:
            label_names.append(name)
ax.set_xlabel('Test subject', fontsize=6)
ax.set_ylabel('Train subject', fontsize=6)
ax.set_title(r'$\mathrm{MAE}_4 - \mathrm{MAE}_{32}\ \mathrm{Time\text{-}Mix}\ \mathrm{across\ subjects}$', fontsize=6)
ax.set_xticks(np.arange(len(infer_subject_names)))
ax.set_yticks(np.arange(len(infer_subject_names)))
ax.set_xticklabels(label_names, rotation=0, fontsize=6)
ax.set_yticklabels(label_names, fontsize=6)
ax.tick_params(axis='both', which='both', length=0)



vmin = -0.01
vmax = 0.01
im.set_clim(vmin, vmax)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelleft=False, left=False, right=False)
cbar.ax.tick_params(labelsize=6)
plt.tight_layout()

In [ ]:
np.sum(mae_matrix_d)

In [ ]:
np.mean([mae_matrix_d[i,i] for i in range(len(mae_matrix_d))])

Show error on anatomy

In [ ]:
import scipy, h5py

reference_anat = scipy.io.loadmat(f'{PATH}/Additional_mat_files/ReferenceBrain.mat')

subject_id = 1

h5_path = f"{PATH}/subject_{subject_id:02d}"
print(h5_path)
h5 = h5py.File(f"{h5_path}/TimeSeries.h5", "r")
abs_ix = h5['absIX']
abs_ix = (abs_ix[0] - 1).astype(int)

mat_path = f"{PATH}/subject_{subject_id:02d}/data_full.mat"
data_struct = scipy.io.loadmat(mat_path)['data'][0, 0]

all_cell_coordinates = data_struct[8]
coordinates = all_cell_coordinates[abs_ix]

plt.figure(figsize=(10, 10))
plt.imshow(np.sum(reference_anat['anat_stack_norm'], axis=-1).T, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
abs_deviation = np.abs(prediction[0]-target[0])
abs_deviation.shape

Error t=1 and t=32

In [ ]:
for i in range(0, 100, 1):
  fig, axs = plt.subplots(2, 1, figsize=(20, 10))
  axs[0].imshow(np.sum(reference_anat['anat_stack_norm'], axis=-1).T, cmap='gray')
  sc0 = axs[0].scatter(
      coordinates[:, 0], coordinates[:, 1],
      c=np.log(np.abs(all_targets_0[i]-all_predictions_0[i])),
      cmap='coolwarm',
      s=0.1,
      vmin=np.log(abs_deviation).min(),
      vmax=np.log(abs_deviation).max()
  )
  axs[0].axis('off')

  axs[1].imshow(np.sum(reference_anat['anat_stack_norm'], axis=-1).T, cmap='gray')
  sc1 = axs[1].scatter(
      coordinates[:, 0], coordinates[:, 1],
      c=np.log(np.abs(all_targets_32[i]-all_predictions_32[i])),
      cmap='coolwarm',
      s=0.1,
      vmin=np.log(abs_deviation).min(),
      vmax=np.log(abs_deviation).max()
  )
  axs[1].axis('off')

  plt.show()
  clear_output(wait=True)
  plt.close()

Cumulative error t=0

In [ ]:
cumulative_abs_error.mean(0)
fig, axs = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
axs.imshow(np.sum(reference_anat['anat_stack_norm'], axis=-1).T, cmap='gray')
sc0 = axs.scatter(
    coordinates[:, 0], coordinates[:, 1],
    c=np.log(cumulative_abs_error.mean(0)),
    cmap='Reds',
    s=0.2,
    alpha=0.5,
)
plt.colorbar(sc0, ax=axs, fraction=0.025, pad=0.04, aspect=30, shrink=1.0).ax.tick_params(labelsize=18)
plt.tight_layout()
axs.axis('off');

Compare with average variability per region (ie, is the model error meaningless in the sense that it is just a reflection of the variability of the data?) 

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
axs.imshow(np.sum(reference_anat['anat_stack_norm'], axis=-1).T, cmap='gray')
sc0 = axs.scatter(
    coordinates[:, 0], coordinates[:, 1],
    c=np.log(target_varibility),
    cmap='coolwarm',
    s=0.2,
)
plt.colorbar(sc0, ax=axs, fraction=0.025, pad=0.04, aspect=30, shrink=1.0).ax.tick_params(labelsize=18)
plt.tight_layout()
axs.axis('off');

In [ ]:
import glob
from zapbench.ts_forecasting import util

path_to_inference = glob.glob("/mnt/storage/misc/zapbench/inference/n_steps_4/mean/**/", recursive=True)[-1]
df = util.get_per_step_metrics_from_directory(path_to_inference,metric='MAE')
df.sort_values('condition')

In [ ]:
for condition in df["condition"].unique():
  print(df[df["condition"] == condition]['MAE'].mean())

In [ ]:
import pandas as pd

from zapbench import constants

df = pd.DataFrame(
    ts_utils.load_json(f'gs://zapbench-release/dataframes/20250131/combined.json'))
df.head()

In [ ]:
df[(df['condition']=='gain') & (df['method']=='mean') & (df['context']==4)]

In [ ]:
idx_list = config.infer_idx_sets[-1]['idx_list']
print(idx_list)

In [ ]:
idx_list = _filter_infer_indices(idx_list, context)

In [ ]:
from zapbench import data_utils

data_utils.adjust_condition_bounds_for_split(
    'test_holdout',
    3078,
    3735,
    4)

In [ ]:
def get_infer_sets(
    num_timesteps_context: int,
    dataset_name: str = constants.DEFAULT_DATASET,
) -> Sequence[dict[str, int | str]]:
  """Get infer sets config with dataset-aware conditions."""
  dataset_config = constants.get_dataset_config(dataset_name)
  conditions_train = dataset_config['conditions_train']
  conditions_holdout = dataset_config['conditions_holdout']

  sets = []
  for condition, split in [(t, 'test') for t in conditions_train] + [
      (t, 'test_holdout') for t in conditions_holdout
  ]:
    inclusive_min, exclusive_max = data_utils.adjust_condition_bounds_for_split(
        split,
        *data_utils.get_condition_bounds(condition, dataset_name=dataset_name),
        num_timesteps_context=num_timesteps_context,
    )
    sets.append({
        'name': f'{split}_condition_{condition}',
        'start_idx': inclusive_min,
        'num_windows': data_utils.get_num_windows(
            inclusive_min, exclusive_max, num_timesteps_context
        ),
    })
  return sets

In [ ]:
infer_sets = get_infer_sets(
  num_timesteps_context=config.timesteps_input_infer
  + config.num_warmup_infer_steps,
  dataset_name=constants.DEFAULT_DATASET,
)

In [ ]:
start_idx = infer_sets[-1]['start_idx']
start_idx

In [ ]:
num_steps=(
    config.num_warmup_infer_steps +
    config.prediction_window_length //
    config.timesteps_output_infer)
num_steps, config.prediction_window_length, config.timesteps_output_infer

Up to this point, everything agrees perfectly... what is going on in infer?

In [ ]:
         predictions, targets = infer(
              model,
              head,
              flax_utils.unreplicate(train_state),
              infer_source,
              start_idx=start_idx + window,
              num_steps=(
                  config.num_warmup_infer_steps +
                  config.prediction_window_length //
                  config.timesteps_output_infer),
              prediction_window_length=config.prediction_window_length,
              infer_key=infer_key,

In [ ]:

start_idx = infer_sets[-1]['start_idx']
num_windows = infer_sets[-1]['num_windows']
for window in range(num_windows):
  real_start_idx = start_idx + window
  t_axis = -2  # Assume shape ...xTxF
  t_in = infer_source[0]['timeseries_input'].shape[t_axis]
  t_out = infer_source[0]['timeseries_output'].shape[t_axis]
  end_idx = min(len(infer_source), real_start_idx + num_steps * t_out)
  series_input_override, carry = None, None
  predictions, targets = [], []
  all_indices = []
  for i, idx in enumerate(range(start_idx, end_idx, t_out)):
    all_indices.append(idx)

In [ ]:
print(all_indices)

In [ ]:
print(idx_list[::32])